# R2.6 T3 — LLM-as-Judge Narrative Evaluation**Model**: gpt-4o-mini · **Temperature**: 0 · **Seed**: 42**Axes**: Coherence (1-5) · Empathy (1-5) · Clinical Fidelity (1-5)Steps:1. Install dependencies2. Set API key from Colab secrets3. Upload `narrative_samples_60.json`4. Run scoring loop5. Aggregate + download

In [ ]:
!pip install openai>=1.40.0 scipy numpy -q

In [ ]:
from google.colab import userdataimport osos.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")print(f'API key loaded (last 4: ...{os.environ["OPENAI_API_KEY"][-4:]})')

In [ ]:
import json, timefrom collections import defaultdictfrom pathlib import Pathimport numpy as npfrom scipy import stats as sp_statsfrom openai import OpenAIfrom google.colab import filesclient = OpenAI()# Upload samplesif not Path("narrative_samples_60.json").exists():    print("Upload narrative_samples_60.json:")    uploaded = files.upload()    fname = list(uploaded.keys())[0]    samples = json.loads(uploaded[fname].decode("utf-8"))else:    with open("narrative_samples_60.json") as f:        samples = json.load(f)print(f"Loaded {len(samples)} samples")

In [ ]:
SYSTEM_PROMPT = """You are an evaluator for a physiotherapy rehabilitation chatbot. Each sample is onechatbot message to a patient, together with the clinical context in which themessage was issued. Score the message on three independent axes (1-5 Likert).Return STRICT JSON with this shape and no additional text:{  "coherence": {"score": <int 1-5>, "reason": "<one sentence>"},  "empathy":   {"score": <int 1-5>, "reason": "<one sentence>"},  "clinical":  {"score": <int 1-5>, "reason": "<one sentence>"}}Axis definitions:- COHERENCE: Does the message follow logically from the context? 1=contradicts, 5=fully consistent.- EMPATHY: Is the emotional register appropriate to pain/fatigue/mood? 1=tone mismatched, 5=well calibrated.- CLINICAL FIDELITY: Does the message correctly describe the task and avoid clinically inappropriate claims? 1=clinically wrong, 5=accurate."""USER_TEMPLATE = """=== CONTEXT ===stream_label: {stream_label}tone: {tone}day: {day}  (of 60)pain_level: {user_pain}  fatigue_level: {user_fatigue}emotional_state_level: {emotional_state_level}task_recommended: {task_name}task_difficulty: {task_difficulty}task_category: {task_category}safety_blocked_today: {safety_blocked_today}scenario_key: {scenario_key}=== CHATBOT MESSAGE (to be scored) ==={prompt_text}Return the JSON object only."""MODEL = "gpt-4o-mini"TEMPERATURE = 0SEED = 42AXES = ["coherence", "empathy", "clinical"]def score_one(sample, temperature=0):    user_msg = USER_TEMPLATE.format(**sample)    resp = client.chat.completions.create(        model=MODEL,        messages=[            {"role": "system", "content": SYSTEM_PROMPT},            {"role": "user", "content": user_msg},        ],        temperature=temperature,        seed=SEED,        response_format={"type": "json_object"},    )    raw = resp.choices[0].message.content    usage = resp.usage    parsed = json.loads(raw)    scores = {}    for ax in AXES:        entry = parsed[ax]        s = int(entry["score"])        assert 1 <= s <= 5, f"{ax} score {s} out of range"        scores[ax] = {"score": s, "reason": entry["reason"]}    return {        "scores": scores, "raw": raw,        "prompt_tokens": usage.prompt_tokens,        "completion_tokens": usage.completion_tokens,    }def score_with_retry(sample):    """API bucket cap=2, format bucket cap=1."""    api_retries = 0    fmt_retries = 0    retries_log = []    for attempt in range(4):        temp = TEMPERATURE if fmt_retries == 0 else 0.1        try:            result = score_one(sample, temperature=temp)            return {"judge_failed": False, "retries": attempt,                    "retries_log": retries_log, **result}        except Exception as e:            err = str(e)            is_api = any(k in err.lower()                         for k in ["timeout","500","502","503","rate","connection"])            if is_api:                if api_retries >= 2: break                api_retries += 1                wait = 2 ** api_retries            else:                if fmt_retries >= 1: break                fmt_retries += 1                wait = 1            retries_log.append({"attempt": attempt+1, "error": err[:200]})            print(f"    Retry {attempt+1}, wait {wait}s")            time.sleep(wait)    return {"judge_failed": True, "retries": api_retries + fmt_retries,            "retries_log": retries_log, "scores": None, "raw": "",            "prompt_tokens": 0, "completion_tokens": 0}print("Scoring functions defined")

In [ ]:
scores_list = []total_pt, total_ct, failed = 0, 0, 0print(f"Scoring {len(samples)} samples with {MODEL}...\n")for i, sample in enumerate(samples):    sid = sample.get("sample_id", f"S{i:02d}")    stream = sample.get("stream_label", "?")    print(f"  [{i+1:02d}/{len(samples)}] {sid} ({stream})", end=" ")    result = score_with_retry(sample)    result["sample_id"] = sid    result["stream_label"] = stream    if result["judge_failed"]:        failed += 1        print("FAILED")    else:        sc = result["scores"]        print(f'coh={sc["coherence"]["score"]} emp={sc["empathy"]["score"]} cli={sc["clinical"]["score"]}')        total_pt += result.get("prompt_tokens", 0)        total_ct += result.get("completion_tokens", 0)    scores_list.append(result)    time.sleep(0.3)usd = total_pt * 0.15/1e6 + total_ct * 0.60/1e6print(f"\nDone: {len(samples)-failed}/{len(samples)} scored, {failed} failed")print(f"  Tokens: {total_pt} prompt + {total_ct} completion")print(f"  Cost: ~${usd:.4f}")if failed >= 6:    print("HOLD: >=6 failures. Do NOT aggregate.")scored_output = {    "config": {"model": MODEL, "temperature": TEMPERATURE, "seed": SEED,               "samples_evaluated": len(samples), "failed": failed},    "cost": {"prompt_tokens": total_pt, "completion_tokens": total_ct,             "usd_estimate": round(usd, 4)},    "scores": scores_list,}with open("narrative_eval_scores.json", "w") as f:    json.dump(scored_output, f, indent=2, ensure_ascii=False)print("Saved: narrative_eval_scores.json")

In [ ]:
# ── Aggregate ──valid = [s for s in scores_list if not s.get("judge_failed")]n_valid = len(valid)overall = {}for ax in AXES:    vals = [s["scores"][ax]["score"] for s in valid]    mean = float(np.mean(vals))    std = float(np.std(vals, ddof=1))    se = std / np.sqrt(len(vals))    ci = sp_stats.t.interval(0.95, df=len(vals)-1, loc=mean, scale=se)    hw = (ci[1] - ci[0]) / 2    overall[ax] = {"mean": round(mean, 2), "ci95_hw": round(hw, 2), "n": len(vals)}by_stream = defaultdict(list)for s in valid:    by_stream[s["stream_label"]].append(s["scores"])stream_bd = {}for stream in sorted(by_stream):    row = {"n": len(by_stream[stream])}    for ax in AXES:        vals = [e[ax]["score"] for e in by_stream[stream]]        row[ax] = round(float(np.mean(vals)), 2)    stream_bd[stream] = rowsep = "=" * 60print(f"\n{sep}")print(f"  Overall (n={n_valid})")for ax in AXES:    o = overall[ax]    print(f'  {ax:12s}: {o["mean"]:.2f} +/- {o["ci95_hw"]:.2f}')print(f"\n  Per-stream:")hdr = f'  {"Stream":<30s} {"coh":>4s} {"emp":>4s} {"cli":>4s}  n'print(hdr)for st in sorted(stream_bd):    r = stream_bd[st]    print(f'  {st:<30s} {r["coherence"]:>4.1f} {r["empathy"]:>4.1f} {r["clinical"]:>4.1f}  {r["n"]}')summary = {"overall": overall, "by_stream_label": stream_bd}with open("narrative_eval_summary.json", "w") as f:    json.dump(summary, f, indent=2, ensure_ascii=False)print(f"\n{sep}")print("Saved: narrative_eval_summary.json")

In [ ]:
# Download resultsfiles.download("narrative_eval_scores.json")files.download("narrative_eval_summary.json")print("Done — copy both JSON files to NarraCare/evaluation/results/")